# Temporal and panel twins

Time series and multi-environment (panel) data get their own twin kinds with their own machinery:
lagged dependencies, latent influence detection, per-environment models, forecasts with
attribution, and interventions that are scheduled in time. This notebook runs all of it live.

| Kind | Kwargs | What it models |
| --- | --- | --- |
| `temporal` | `time=` | one time series with lagged causal structure |
| `multi-environment-static` | `entity=` | the same system observed across environments |
| `multi-environment-temporal` | `time=` + `entity=` | a panel: many environments, each a time series |

In [ ]:
import os
import numpy as np
import pandas as pd

import rootcause as rc

rc.login(base_url=os.environ.get("ROOTCAUSE_BASE_URL", "https://platform.rootcause.ai"))

## A panel: three stores, thirty months

Ground truth: price suppresses demand, demand carries momentum (a lag), seasonality moves both,
and each store runs at its own scale. Long format: one row per store per month.

In [ ]:
rng = np.random.default_rng(11)
months = pd.date_range("2024-01-01", periods=30, freq="MS")
stores = {"london": 1.0, "paris": 0.8, "berlin": 1.25}
rows = []
for store, scale in stores.items():
    demand_prev = 100.0 * scale
    for i, month in enumerate(months):
        season = 12 * np.sin(2 * np.pi * (i % 12) / 12)
        price = 20 + 2 * np.sin(2 * np.pi * (i % 12) / 12 + 1) + rng.normal(0, 0.5)
        demand = 0.55 * demand_prev + 60 * scale - 2.4 * price + season + rng.normal(0, 4)
        revenue = price * demand * 0.1 + rng.normal(0, 3)
        rows.append({"month": month.strftime("%Y-%m-%d"), "store": store,
                     "price": round(price, 2), "demand": round(demand, 1), "revenue": round(revenue, 1)})
        demand_prev = demand
panel = pd.DataFrame(rows)
panel.head()

## Discovery finds more than edges

`time=` and `entity=` make this a panel-temporal twin. Note the graph: alongside the causal
edges, discovery surfaced a **latent influence**, a hidden common cause it detected in the data
but could not name, and it attributes the store-level differences to the environment itself.

In [ ]:
graph = rc.discover(panel, time="month", entity="store", force=True)
graph.edges

In [ ]:
twin = graph.train()
twin

## Per-environment sampling

Panel twins hold one model per environment. Sampling narrows with `environments=` and the
returned frame carries an `environment` column; seeds derive stable per-environment children,
so comparisons are deterministic.

In [ ]:
draws = twin.sample(n=500, environments=["london", "berlin"], seed=3)
draws.to_frame().groupby("environment").mean(numeric_only=True).round(1)

## Forecasts carry their reasoning

Each forecast step comes with bounds and an attribution: how much of the prediction is trend,
season, and each causal parent (with lags). `aggregate="sum"` adds a combined series across
environments; `origin_timestamp` anchors backtests.

In [ ]:
fc = twin.forecast(horizon=6, targets=["revenue"], aggregate="sum")
fc.to_frame()[["environment", "timestamp", "prediction", "lowerBound", "upperBound"]].head(8).round(1)

In [ ]:
fc.to_frame().loc[0, "attribution"]

## Interventions scheduled in time

`rc.at` wraps any intervention value with when it applies: `persistent=True` from the first
step onwards, `duration_steps=` for a limited window, `timestamp=` for a specific start.
Here: a permanent 10 percent price cut, in London only.

In [ ]:
result = twin.intervene(
    {"price": rc.at(rc.pct(-10), persistent=True)},
    outcomes=["revenue"],
    environments=["london"],
)
result

## Working with a subset of environments

`twin.environments` lists what the panel actually holds — one row per
environment with its sample size:

In [ ]:
twin.environments

`twin.env(...)` pins a handle to some of them. Its `graph` re-aggregates the
causal adjacency over just those environments — edges carry `agreementRate`,
the share of the subset's environments in which discovery found the
relationship — and `adjacency(agreement_threshold=...)` turns the edge-survival
knob (default 0.5). Every simulation on the handle is scoped automatically.

In [ ]:
eu = twin.env("london", "berlin")
adjacency = eu.graph
print(f"{adjacency.attrs['envCount']} of {adjacency.attrs['totalEnvCount']} environments, "
      f"threshold {adjacency.attrs['agreementThreshold']}")
adjacency[["source", "target", "strength", "agreementRate"]]

Simulations on the handle run only in the subset — same verbs, pre-scoped.
The intervention below asks what a price cut does to revenue in London and
Berlin, leaving Paris untouched:

In [ ]:
eu.intervene({"price": rc.at(rc.pct(-10), persistent=True)}, outcomes=["revenue"])

In [ ]:
eu.forecast(horizon=3, targets=["revenue"]).to_frame()[
    ["environment", "timestamp", "prediction", "lowerBound", "upperBound"]
].round(1)

Environments can also be selected **by their data** instead of by name — `where=`
filters on any twin column through per-environment statistics, so "the
high-revenue stores" needs no hand-maintained list. A tuple is
`(column, op, value)` for constant-per-environment columns, or
`(column, reduce, op, value)` with reduce one of `avg`, `min`, `max`, or `any`;
`.environments` shows exactly what matched before you run anything:

In [ ]:
high_rev = twin.env(where=[("revenue", "avg", ">", 100)])
high_rev.environments

In [ ]:
high_rev.forecast(horizon=2, targets=["revenue"]).to_frame()[
    ["environment", "timestamp", "prediction"]
].round(1)

## Monthly refresh: assimilate instead of retrain

When next month's rows arrive, the model doesn't need rebuilding — extend the twin's
source and fold the new rows into the fitted model with `update()`. It finishes with a
status, never an error: `committed` (rows folded in), `up_to_date` (nothing new), or
`retrain_required` (the model can't take these rows incrementally — call `twin.retrain()`).

Static and temporal twins assimilate out of the box. Panel twins need the v2 panel
engine (an opt-in in the twin builder); on anything else `update()` simply reports
`retrain_required`, and `twin.update_eligibility` tells you in advance. London's series
as its own temporal twin:

In [ ]:
london = panel[panel["store"] == "london"][["month", "price", "demand", "revenue"]].reset_index(drop=True)
monthly = rc.discover(london, time="month", force=True).train()
monthly

Two months pass. Extend the source with the new rows and update — the transcript is
the whole loop:

In [ ]:
source = monthly.source
current = source.to_frame()
last_month = pd.to_datetime(current["month"]).max()
demand_prev = current.sort_values("month")["demand"].iloc[-1]

rows = []
for month in pd.date_range(last_month + pd.offsets.MonthBegin(1), periods=2, freq="MS"):
    j = (month.year - 2024) * 12 + (month.month - 1)
    season = 12 * np.sin(2 * np.pi * (j % 12) / 12)
    price = 20 + 2 * np.sin(2 * np.pi * (j % 12) / 12 + 1) + rng.normal(0, 0.5)
    demand = 0.55 * demand_prev + 60 - 2.4 * price + season + rng.normal(0, 4)
    rows.append({"month": month.strftime("%Y-%m-%d"),
                 "price": round(price, 2), "demand": round(demand, 1),
                 "revenue": round(price * demand * 0.1 + rng.normal(0, 3), 1)})
    demand_prev = demand

source.extend(pd.DataFrame(rows))
result = monthly.update()
result

Running it again with nothing new in the source is how a scheduled job stays honest —
the second call is a cheap no-op:

In [ ]:
monthly.update()

The refreshed model forecasts onwards from the assimilated months:

In [ ]:
fc = monthly.forecast(horizon=3, targets=["revenue"])
fc.to_frame()[["timestamp", "prediction", "lowerBound", "upperBound"]].round(1)

The same scheduling works on plain temporal twins, and everything else from the
[quickstart](quickstart.ipynb) applies unchanged: `save`, `load_twin`, `console`, and the
ontology all understand temporal and panel twins.